# Lecture 2 — Class Exercise
# Bar Charts: World Happiness Report 2023

---

> **Your task:** Create **2 polished bar charts** using the World Happiness Report dataset.  
> **Push to:** `week02/lecture02_exercise.ipynb` in **your own GitHub repo** before the end of class.

---

### Rules to keep in mind 💡
1. Every bar chart **must have a zero baseline** — no exceptions
2. Every chart **must have an insight title**, not a topic title
3. Aim for **professional quality** — clean background, readable font, no clutter
4. Horizontal bars for long category names

---


# Setup — Run this cell first

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Kaggle-friendly path: change only this line if your dataset has a different path.
DATA_PATH = '../data/world_happiness_2023.csv'
df = pd.read_csv(DATA_PATH)

# Keep the expected lecture column names.
df.columns = ['Country','Region','Happiness_Score','GDP','Social_Support',
              'Life_Expectancy','Freedom','Generosity','Corruption']

print(f'Dataset: {len(df)} countries, {len(df.columns)} columns')
display(df.head())


In [ ]:
import pandas as pd
import numpy as np

# World Happiness Report 2023 — representative data
# Source: https://www.kaggle.com/datasets/ajaypalsinghlo/world-happiness-report-2023

df = pd.read_csv('../data/world_happiness_2023.csv')
df.columns = ['Country','Region','Happiness_Score','GDP','Social_Support',
              'Life_Expectancy','Freedom','Generosity','Corruption']


print(f"Dataset: {len(df)} countries, {len(df.columns)} columns")
print(df.head())


In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# Explore the dataset before you start
print("Regions in dataset:")
print(df['Region'].value_counts())
print("\nScore range:", df['Happiness_Score'].min(), "–", df['Happiness_Score'].max())
print("\nBottom 10 countries:")
print(df.nsmallest(10, 'Happiness_Score')[['Country','Region','Happiness_Score']])


---
## Task 1 — Regional Comparison Bar Chart

**What to build:** A horizontal bar chart showing the **average happiness score by region**, sorted from highest to lowest.

**Requirements:**
- Horizontal orientation (category names are long)
- Sorted by score, descending (so the happiest region is at the top)
- Zero baseline on x-axis
- At least one design choice that goes beyond the Plotly default (colour, annotation, labels, etc.)
- An insight title that answers: *which region stands out and why does it matter?*

**Hint:** Use `df.groupby('Region')['Happiness_Score'].mean()` to compute the averages.


In [ ]:
# Task 1: Regional comparison bar chart
# -------------------------------------

region_avg = (df.groupby('Region')['Happiness_Score']
              .mean()
              .reset_index()
              .sort_values('Happiness_Score', ascending=True))

fig = px.bar(
    region_avg,
    x='Happiness_Score',
    y='Region',
    orientation='h',
    text=region_avg['Happiness_Score'].round(2),
    title='Western Europe Leads Regional Happiness, While Lower-Ranked Regions Face a Clear Well-Being Gap',
    labels={'Happiness_Score': 'Average Happiness Score', 'Region': 'Region'},
)

fig.update_traces(marker_color='#2E86AB', textposition='outside', cliponaxis=False)
fig.update_layout(
    xaxis=dict(range=[0, max(region_avg['Happiness_Score'].max() * 1.18, 10)], title='Average Happiness Score'),
    yaxis=dict(title=''),
    template='plotly_white',
    margin=dict(l=180, r=80, t=90, b=60),
    font=dict(size=13),
)
fig.show()


---
## Task 2 — Bottom vs. Top: A Contrast Story

**What to build:** A bar chart that highlights the **gap between the happiest and least happy countries**, focusing on a specific insight.

**Requirements:**
- Show the **top 8 AND bottom 8 countries** together (16 bars total)
- Use **colour** to distinguish the two groups (not Plotly's default rainbow)
- Add a **visual separator or annotation** that emphasises the gap
- Insight title that tells the story of the gap

**Hint:** Use `pd.concat([df.nlargest(8,'Happiness_Score'), df.nsmallest(8,'Happiness_Score')])` to get both groups.

**Stretch goal:** Add a vertical reference line showing the global average.


In [ ]:
# Task 2: Top 8 vs. Bottom 8 contrast
# ------------------------------------

top8 = df.nlargest(8, 'Happiness_Score').copy()
top8['Group'] = 'Top 8'
bottom8 = df.nsmallest(8, 'Happiness_Score').copy()
bottom8['Group'] = 'Bottom 8'

combined = pd.concat([bottom8, top8]).sort_values('Happiness_Score')
global_avg = df['Happiness_Score'].mean()
print(f'Global average: {global_avg:.2f}')

fig = px.bar(
    combined,
    x='Happiness_Score',
    y='Country',
    orientation='h',
    color='Group',
    color_discrete_map={'Top 8': '#2E8B57', 'Bottom 8': '#C44E52'},
    text=combined['Happiness_Score'].round(2),
    title='The Happiest Countries Score Far Above the Least Happy Countries',
    labels={'Happiness_Score': 'Happiness Score', 'Country': 'Country'},
)

fig.add_vline(
    x=global_avg, line_dash='dash', line_width=2,
    annotation_text=f'Global average: {global_avg:.2f}',
    annotation_position='top'
)
fig.update_traces(textposition='outside', cliponaxis=False)
fig.update_layout(
    xaxis=dict(range=[0, max(combined['Happiness_Score'].max() * 1.18, 10)], title='Happiness Score'),
    yaxis=dict(title=''),
    template='plotly_white',
    margin=dict(l=170, r=100, t=100, b=60),
    font=dict(size=13),
    legend_title_text='Group'
)
fig.show()


---
## Done? Stretch Goal

If you finish both tasks with time to spare, try this:

**Task 3 (stretch):** Build a **grouped bar chart** comparing 2 sub-factors (e.g. `GDP_per_capita` and `Freedom`) across the 5 most populated regions. Use colour meaningfully and write an insight title.

Regions to include: `'Western Europe'`, `'Latin America'`, `'East Asia'`, `'Sub-Saharan Africa'`, `'South Asia'`


In [ ]:
# Stretch goal — grouped bar chart
# ---------------------------------

regions = ['Western Europe', 'Latin America', 'East Asia', 'Sub-Saharan Africa', 'South Asia']

subset = df[df['Region'].isin(regions)].copy()
region_factors = (subset.groupby('Region')[['GDP', 'Freedom']]
                  .mean()
                  .reindex(regions)
                  .reset_index())

long_df = region_factors.melt(
    id_vars='Region',
    value_vars=['GDP', 'Freedom'],
    var_name='Factor',
    value_name='Average Value'
)

fig = px.bar(
    long_df,
    x='Region',
    y='Average Value',
    color='Factor',
    barmode='group',
    color_discrete_map={'GDP': '#4C78A8', 'Freedom': '#F58518'},
    title='Western Europe Shows the Strongest Combination of Economic Strength and Freedom',
    labels={'Average Value': 'Average Value', 'Region': 'Region'},
)
fig.update_layout(
    yaxis=dict(range=[0, max(long_df['Average Value'].max() * 1.18, 1)], title='Average Value'),
    template='plotly_white',
    margin=dict(l=70, r=40, t=90, b=100),
    font=dict(size=13),
)
fig.show()
